# 102 - Ollama: modelos de lenguaje locales

Ollama permite ejecutar LLMs localmente. En clase es útil para practicar sin coste de API y sin depender siempre de servicios cloud.

El notebook usa un modo simulado si Ollama no está instalado o si el servicio local no está arrancado.

## Preparación para Colab

En Colab se puede instalar el paquete Python `ollama`, pero el servicio local de Ollama normalmente debe estar ejecutándose en una máquina propia. Por eso el notebook mantiene un fallback simulado.

In [ ]:
import importlib.util
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))

if IN_COLAB and importlib.util.find_spec("ollama") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ollama"])
    print("Paquete Python ollama instalado.")
else:
    print("Instalación automática omitida o paquete ya disponible.")

## 1. Cliente de modelo local

Esta celda intenta usar Ollama real. Si no puede, devuelve una respuesta determinista para que el resto del notebook siga siendo ejecutable.

In [ ]:
def call_local_llm(prompt, model="llama3"):
    try:
        import ollama
        response = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
        )
        return {"backend": "ollama", "text": response["message"]["content"]}
    except Exception as exc:
        simulated = f"[Simulación local/{model}] Respuesta breve para: {prompt}"
        return {"backend": "simulado", "text": simulated, "reason": str(exc)}

result = call_local_llm("Explica qué es el sobreajuste en una frase.")
print(result)

## 2. Conversación con historial

Un chat local mantiene una lista de mensajes. En una aplicación real esa lista se enviaría a Ollama en cada turno.

In [ ]:
conversation = []

def chat_turn(user_message, model="llama3"):
    conversation.append({"role": "user", "content": user_message})
    prompt = "\n".join(f"{m['role']}: {m['content']}" for m in conversation[-4:])
    response = call_local_llm(prompt, model=model)
    conversation.append({"role": "assistant", "content": response["text"]})
    return response

print(chat_turn("Me llamo Ana."))
print(chat_turn("¿Cómo me llamo?"))

## 3. Ollama como generador en un RAG mínimo

Ollama puede ser el modelo generador final de un sistema RAG construido con LangChain o LlamaIndex. Aquí simulamos la recuperación y usamos `call_local_llm` como generador.

In [ ]:
DOCUMENTS = [
    {"title": "Ollama", "text": "Ollama ejecuta modelos de lenguaje en local."},
    {"title": "RAG", "text": "RAG combina recuperación de documentos y generación de respuestas."},
    {"title": "Privacidad", "text": "Los modelos locales pueden evitar enviar datos a servicios externos."},
]

def retrieve_context(question):
    q = question.lower()
    return [doc for doc in DOCUMENTS if any(word in doc["text"].lower() for word in q.split())][:2]

def answer_with_local_model(question):
    context = retrieve_context(question)
    context_text = " ".join(doc["text"] for doc in context)
    prompt = f"Contexto: {context_text}\nPregunta: {question}\nResponde usando solo el contexto."
    answer = call_local_llm(prompt)
    return {"context": context, "answer": answer}

print(answer_with_local_model("¿Por qué usar un modelo local?"))

## 4. Registro de configuración

Cuando se usa Ollama en experimentos, conviene registrar modelo, proveedor y si la inferencia es local. Esto se puede conectar con MLflow.

In [ ]:
ollama_run = {
    "provider": "ollama",
    "model": "llama3",
    "local": True,
    "use_case": "chat_or_rag",
}
print(ollama_run)

## Reto adicional

Si tienes Ollama instalado, descarga un modelo pequeño con `ollama pull llama3` o similar y compara la respuesta real con la simulada.

Para profundizar: consulta `Documentacion/Ollama_Documentacion.md`.